In [3]:
import pandas as pd
import os
import time
import requests
from datetime import datetime, timezone

def load_csv_from_link(link):
    resp = requests.get(link, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    if isinstance(data, list):
        data = data[0]

    if isinstance(data, dict):
        df = pd.json_normalize(data)
        return df

def extract_plays_from_sportsdata_pbp(df: pd.DataFrame) -> pd.DataFrame:
    """
    Takes the SportsDataIO play-by-play dataframe where 'Plays' is a nested column
    and returns one row per play, with game metadata attached.
    """
    if df.empty:
        return pd.DataFrame()

    if "Plays" not in df.columns:
        raise ValueError("No 'Plays' column found.")

    all_play_dfs = []

    for _, row in df.iterrows():
        plays = row["Plays"]
        if not isinstance(plays, list) or len(plays) == 0:
            continue

        # Keep score/game-level metadata
        game_meta = {
            col: row[col]
            for col in df.columns
            if col.startswith("Score.")
        }

        plays_df = pd.json_normalize(plays)

        for k, v in game_meta.items():
            plays_df[k] = v

        all_play_dfs.append(plays_df)

    if not all_play_dfs:
        return pd.DataFrame()

    out = pd.concat(all_play_dfs, ignore_index=True, sort=False)

    return out

df = load_csv_from_link(link='http://archive.sportsdata.io/v3/nfl/pbp/json/playbyplay/19055/2025-09-08-03-02.json')
extracted_df = extract_plays_from_sportsdata_pbp(df)
extracted_df

,PlayID,QuarterID,QuarterName,Sequence,TimeRemainingMinutes,TimeRemainingSeconds,PlayTime,Updated,Created,Team,...,Score.StadiumDetails.StadiumID,Score.StadiumDetails.Name,Score.StadiumDetails.City,Score.StadiumDetails.State,Score.StadiumDetails.Country,Score.StadiumDetails.Capacity,Score.StadiumDetails.PlayingSurface,Score.StadiumDetails.GeoLat,Score.StadiumDetails.GeoLong,Score.StadiumDetails.Type
0,612889,13482,1,1,15,0,2025-09-07T13:02:59,2025-09-08T02:52:29,2025-09-07T13:03:40,TB,...,45,Mercedes-Benz Stadium,Atlanta,GA,USA,71000,Artificial,33.755556,-84.401,RetractableDome
1,612898,13482,1,2,14,54,2025-09-07T13:03:54,2025-09-08T02:52:29,2025-09-07T13:04:24,ATL,...,45,Mercedes-Benz Stadium,Atlanta,GA,USA,71000,Artificial,33.755556,-84.401,RetractableDome
2,612906,13482,1,3,14,10,2025-09-07T13:04:36,2025-09-08T02:52:29,2025-09-07T13:05:06,ATL,...,45,Mercedes-Benz Stadium,Atlanta,GA,USA,71000,Artificial,33.755556,-84.401,RetractableDome
3,612919,13482,1,4,13,26,2025-09-07T13:05:24,2025-09-08T02:52:29,2025-09-07T13:06:02,ATL,...,45,Mercedes-Benz Stadium,Atlanta,GA,USA,71000,Artificial,33.755556,-84.401,RetractableDome
4,612924,13482,1,5,13,14,2025-09-07T13:06:18,2025-09-08T02:52:29,2025-09-07T13:06:33,ATL,...,45,Mercedes-Benz Stadium,Atlanta,GA,USA,71000,Artificial,33.755556,-84.401,RetractableDome
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186,614376,13504,4,187,0,6,2025-09-07T16:24:15,2025-09-08T02:52:29,2025-09-07T16:25:29,ATL,...,45,Mercedes-Benz Stadium,Atlanta,GA,USA,71000,Artificial,33.755556,-84.401,RetractableDome
187,614381,13504,4,188,0,6,2025-09-07T16:26:10,2025-09-08T02:52:29,2025-09-07T16:26:31,ATL,...,45,Mercedes-Benz Stadium,Atlanta,GA,USA,71000,Artificial,33.755556,-84.401,RetractableDome
188,614390,13504,4,189,0,2,2025-09-07T16:27:04,2025-09-08T02:52:29,2025-09-07T16:27:26,TB,...,45,Mercedes-Benz Stadium,Atlanta,GA,USA,71000,Artificial,33.755556,-84.401,RetractableDome
189,614393,13504,4,190,0,0,2025-09-07T16:27:04,2025-09-08T02:52:29,2025-09-07T16:27:56,TB,...,45,Mercedes-Benz Stadium,Atlanta,GA,USA,71000,Artificial,33.755556,-84.401,RetractableDome


In [4]:
df = load_csv_from_link(link='http://archive.sportsdata.io/v3/nfl/odds/json/livegameoddslinemovement/19055/2025-09-08-02-59.json')

def convert_est_columns_to_utc(df: pd.DataFrame) -> pd.DataFrame:
    """
    SportsDataIO archive timestamps appear to be in US Eastern time.
    This converts all likely date/time columns to UTC.

    Uses America/New_York instead of fixed EST so daylight saving time is handled correctly.
    """
    out = df.copy()

    date_cols = [
        col for col in out.columns
        if any(x in col.lower() for x in ["date", "time", "created", "updated"])
    ]

    for col in date_cols:
        parsed = pd.to_datetime(out[col], errors="coerce")

        # Skip columns that clearly are not datetime-like
        if parsed.notna().sum() == 0:
            continue

        # If timezone-naive, assume America/New_York, then convert to UTC
        if parsed.dt.tz is None:
            out[col] = (
                parsed
                .dt.tz_localize("America/New_York", ambiguous="NaT", nonexistent="shift_forward")
                .dt.tz_convert("UTC")
            )
        else:
            out[col] = parsed.dt.tz_convert("UTC")

    return out

def extract_liveodds_from_sportsdata_pbp(df: pd.DataFrame) -> pd.DataFrame:
    """
    Takes the SportsDataIO live odds dataframe where 'LiveOdds' is a nested column
    and returns one row per odd update, with game metadata attached.
    """
    if df.empty:
        return pd.DataFrame()

    if "LiveOdds" not in df.columns:
        raise ValueError("No 'LiveOdds' column found.")

    all_odds_dfs = []

    for _, row in df.iterrows():
        odds = row["LiveOdds"]
        if not isinstance(odds, list) or len(odds) == 0:
            continue

        # Keep score/game-level metadata
        game_meta = {
            col: row[col]
            for col in df.columns
            if col.startswith("Score.")
        }

        odds_df = pd.json_normalize(odds)

        for k, v in game_meta.items():
            odds_df[k] = v

        all_odds_dfs.append(odds_df)

    if not all_odds_dfs:
        return pd.DataFrame()

    out = pd.concat(all_odds_dfs, ignore_index=True, sort=False)
    out = convert_est_columns_to_utc(out)

    return out

extracted_odds = extract_liveodds_from_sportsdata_pbp(df)


output_path = "data/KXNFLGAME-25SEP07TBATL.csv"

if not extracted_odds.empty:
    write_header = not os.path.exists(output_path)
    extracted_odds.to_csv(output_path, mode="a", header=write_header, index=False)
